<a href="https://colab.research.google.com/github/jasecolino/Jase/blob/main/LEX_and_YACC_CompilerLab10-JAC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LEX and YACC Compiler in Colab

Drawbacks:
* Regular interrupts (Ctrl+D, Ctrl+C) for shell won't work in Colab while inputting for program.
<br>Workaround: Store your inputs in a txt file and pass it to the program.

In [51]:
#@title Install *prerqeuisites* (run this cell first to work on LEX/YACC)
!sudo apt install flex bison

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
bison is already the newest version (2:3.8.2+dfsg-1build1).
flex is already the newest version (2.6.4-8build2).
0 upgraded, 0 newly installed, 0 to remove and 5 not upgraded.


In [52]:
#@title Lex program to count identifiers
%%writefile program.l

%{
    int count=0;
%}
%%
[a-zA-Z_][a-zA-Z0-9_]* { count++; }
.|\n ;
%%

int yywrap(void) {
    return 1;
}

int main(int argc, char *argv[]) {
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }
    yyin=fopen(argv[1],"r");

    yylex();
    printf("number of identifiers = %d\n", count);

    fclose(yyin);
    return 0;
}

Overwriting program.l


In [53]:
#@title Create a file with one possible identifier per line to check
%%writefile program.txt
123
num
num2
2num
123xyz
_123
_num
num_123
123_num

Overwriting program.txt


In [54]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

# Remove problematic comments from program.l before processing with lex
sed -i '/^\s*\/\*.*\*\/\s*$/d' program.l

# Remove leading whitespace from the Lex rule line
sed -i '/\[a-zA-Z_][a-zA-Z0-9_]*/s/^[[:blank:]]*//' program.l

lex -l program.l
gcc lex.yy.c
./a.out program.txt

number of identifiers = 8


In [55]:
#@title Writing Lex program to find email addresses
%%writefile program.l
%{
#include <stdio.h>
%}

%%
[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,4} { printf("Found email ID: %s\n", yytext); }
.|\n ;
%%

int main(int argc, char **argv) {
    if(argc!=2){
        printf("Usage:\n\t./a.out <FILENAME>\n");
        exit(0);
    }
    yyin=fopen(argv[1],"r");
    yylex();
    return 0;
}

int yywrap() {
    return 1;
}

Overwriting program.l


In [56]:
#@title Create a file with several possible email addresses
%%writefile program.txt
no way kruse@juniata.edu
helpe@nune LCKruse@yahoo.com
four gwk@cfm.brown.edu
kruseb@msn.com five
six
seven
Jerry.Kruse@juniata.edu and more
nine Jerry-Kruse@juniata.edu
ten
help@juniata.edu and hr@ibm.co.de
jerry@.com @yahoo.com first@lame

Overwriting program.txt


In [57]:
#@title Shell Execution (you can rewrite the commands as per your need, eg. if you want to include a file as an input)
%%shell

lex email_fix.l
gcc lex.yy.c -o email_finder
./email_finder test_emails.txt

Found email ID: kruse@juniata.edu
Found email ID: LCKruse@yahoo.com
Found email ID: gwk@cfm.brown.edu
Found email ID: kruseb@msn.com
Found email ID: Jerry.Kruse@juniata.edu
Found email ID: Jerry-Kruse@juniata.edu
Found email ID: help@juniata.edu
Found email ID: hr@ibm.co.de
